In [ ]:
import pandas as pd
import numpy as np
import re
import time
from difflib import SequenceMatcher

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

import requests
from bs4 import BeautifulSoup

In [3]:
phenos = pd.read_csv("/Users/prince/philly-tree-mapper/data/processed/philly_species_pheno_DOY.csv")

In [4]:
phenos

,Unnamed: 0,tree_name,scientific_name,common_name,Genus,Species,plant_code,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,...,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL,bloom_range_DOY,fruit_range_DOY,Type
0,0,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,No,Yellow,No,...,abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN,"[244, 245, 246, 247, 248, 249, 250, 251, 252, ...",Needled evergreen
1,1,Abies fraseri - fraser fir,abies fraseri,fraser fir,Abies,fraseri,ABFR,No,Purple,No,...,abies fraseri,Non-flowering,N/a,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 325, ...",Needled evergreen
2,2,Acer ginnala - amur maple,acer ginnala,amur maple,Acer,ginnala,ACGI,Yes,White,No,...,acer ginnala,N/a,N/a,N/a,N/a,no results container,NaN,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 172, ...","[152, 153, 154, 155, 156, 157, 158, 159, 160, ...",NO_URL
3,3,Acer negundo - boxelder,acer negundo,boxelder,Acer,negundo,ACNE2,Yes,White,No,...,acer negundo,March to April,Greenish-yellow,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 7...","[152, 153, 154, 155, 156, 157, 158, 159, 160, ...",Tree
4,4,Acer nigrum - black maple,acer nigrum,black maple,Acer,nigrum,ACNI5,Yes,Yellow,No,...,acer nigrum,N/a,N/a,N/a,N/a,no results container,NaN,"[91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101,...","[152, 153, 154, 155, 156, 157, 158, 159, 160, ...",NO_URL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,289,Malus species - sugar tyme crabapple,malus species,sugar tyme crabapple,Malus,species,NaN,NaN,NaN,NaN,...,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NO_URL
290,290,Rosa species - other rose,rosa species,other rose,Rosa,species,NaN,NaN,NaN,NaN,...,rosa species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NO_URL
291,291,Acer henryii – henrys maple,acer henryii,henrys maple,Acer,henryii,NaN,NaN,NaN,NaN,...,acer henryii,N/a,N/a,N/a,N/a,no results container,NaN,NaN,NaN,NO_URL
292,292,Malus species - indian summer crabapple,malus species,indian summer crabapple,Malus,species,NaN,NaN,NaN,NaN,...,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NO_URL


In [ ]:

# ── constants ─────────────────────────────────────────────────────────────────
SEARCH_URL       = "https://www.missouribotanicalgarden.org/plantfinder/plantfindersearch.aspx"
CHECKPOINT_PATH  = "/Users/prince/philly-tree-mapper/data/processed/mobot_rescrape_checkpoint.csv"
OUTPUT_PATH      = "/Users/prince/philly-tree-mapper/data/processed/philly_species_pheno_v2.csv"

# ── search helpers (Selenium) ─────────────────────────────────────────────────

_CULTIVAR_RE = re.compile(r"'([^']+)'")

def _norm(s):
    return re.sub(r"[^a-z0-9\s]", " ", s.lower()).strip()

def _score(query, title):
    """Similarity between a tree_name query and a search-result title.
    Penalizes results whose cultivar name does not appear in the query,
    so a plain species page is always preferred over a cultivar page
    when the query names no cultivar.
    """
    clean = re.sub(r"\s*[-–]\s*plant finder.*$", "", title, flags=re.IGNORECASE)
    base  = SequenceMatcher(None, _norm(query), _norm(clean)).ratio()
    m = _CULTIVAR_RE.search(clean)
    if m and _norm(m.group(1)) not in _norm(query):
        base *= 0.6
    return base

def get_search_results(driver, tree_name, timeout=15):
    """
    Submit tree_name to the MOBOT search box and return
    a list of (title, url) tuples from the results panel.
    """
    driver.get(SEARCH_URL)
    box = WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.ID, "GoogleQuery"))
    )
    box.clear()
    box.send_keys(tree_name)
    driver.find_element(By.ID, "GoogleSearch").click()

    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.gsc-webResult"))
        )
        time.sleep(0.8)   # let any remaining results render
    except Exception:
        return []

    pairs = []
    for r in driver.find_elements(By.CSS_SELECTOR, "div.gsc-webResult"):
        try:
            a = r.find_element(By.CSS_SELECTOR, "a.gs-title")
            title = a.text.strip()
            url   = a.get_attribute("data-ctorig") or ""
            if "PlantFinderDetails" in url and title:
                pairs.append((title, url))
        except Exception:
            continue
    return pairs

def pick_best(tree_name, pairs):
    """Return (title, url, score) of the highest-scoring result."""
    if not pairs:
        return None, None, 0.0
    scored = sorted(
        ((_score(tree_name, t), t, u) for t, u in pairs),
        reverse=True,
    )
    score, title, url = scored[0]
    return title, url, round(score, 4)


# ── detail-page scraper (requests + BS4) ──────────────────────────────────────

_SESSION = requests.Session()
_SESSION.headers.update({"User-Agent": "Mozilla/5.0 (research data collection)"})

_FALL_RE      = re.compile(r"[^.]*\b(?:fall|autumn)\s+(?:color|colour|foliage)\b[^.]*\.", re.IGNORECASE)
_LEAF_TYPE_RE = re.compile(r"\b(deciduous|evergreen|semi-evergreen|semi‑evergreen|semievergreen)\b", re.IGNORECASE)
_SEX_RE       = re.compile(r"\b(dioecious|monoecious|male|female|pistillate|staminate)\b", re.IGNORECASE)

def _row_text(soup, div_id):
    tag = soup.find("div", id=div_id)
    if tag is None:
        return None
    text = tag.get_text(" ", strip=True)
    return text.split(":", 1)[1].strip() if ":" in text else None

def _unlabeled(soup, label):
    """Collect all unlabeled rows in .column-right that begin with label:"""
    hits = []
    for div in soup.select("div.column-right div.row"):
        if div.get("id"):
            continue
        text = div.get_text(" ", strip=True)
        if text.lower().startswith(label.lower() + ":"):
            val = text.split(":", 1)[1].strip()
            if val:
                hits.append(val)
    return ", ".join(hits) if hits else None

def _parse_noteworthy(text):
    fall_match = _FALL_RE.search(text)
    fall_color = fall_match.group(0).strip() if fall_match else None

    leaf_hits = list(dict.fromkeys(m.group(1).lower() for m in _LEAF_TYPE_RE.finditer(text)))
    leaf_type  = ", ".join(leaf_hits) if leaf_hits else None

    sex_hits  = list(dict.fromkeys(m.group(1).lower() for m in _SEX_RE.finditer(text)))
    sex_info  = ", ".join(sex_hits) if sex_hits else None

    return fall_color, leaf_type, sex_info

def scrape_detail(url):
    """Fetch a PlantFinderDetails page and return a dict of scraped fields."""
    try:
        resp = _SESSION.get(url, timeout=15)
        resp.raise_for_status()
    except requests.RequestException:
        return None

    soup = BeautifulSoup(resp.text, "html.parser")
    nw_div  = soup.find("div", id="MainContentPlaceHolder_NoteworthyRow")
    nw_text = nw_div.get_text(" ", strip=True) if nw_div else ""
    fall_color, leaf_type, sex_info = _parse_noteworthy(nw_text)

    return {
        "mobot_common_name":   _row_text(soup, "MainContentPlaceHolder_CommonNameRow"),
        "mobot_type":          _row_text(soup, "MainContentPlaceHolder_TypeRow"),
        "mobot_family":        _row_text(soup, "MainContentPlaceHolder_FamilyRow"),
        "mobot_zone":          _row_text(soup, "MainContentPlaceHolder_ZoneRow"),
        "mobot_height":        _row_text(soup, "MainContentPlaceHolder_HeightRow"),
        "mobot_spread":        _row_text(soup, "MainContentPlaceHolder_SpreadRow"),
        "mobot_bloom_time":    _row_text(soup, "MainContentPlaceHolder_BloomTimeRow"),
        "mobot_bloom_desc":    _row_text(soup, "MainContentPlaceHolder_ColorTextRow"),
        "mobot_sun":           _row_text(soup, "MainContentPlaceHolder_SunRow"),
        "mobot_water":         _row_text(soup, "MainContentPlaceHolder_WaterRow"),
        "mobot_maintenance":   _row_text(soup, "MainContentPlaceHolder_MaintenanceRow"),
        "mobot_suggested_use": _unlabeled(soup, "Suggested Use"),
        "mobot_flower":        _unlabeled(soup, "Flower"),
        "mobot_tolerate":      _unlabeled(soup, "Tolerate"),
        "mobot_fall_color":    fall_color,
        "mobot_leaf_type":     leaf_type,
        "mobot_sex_info":      sex_info,
        "mobot_noteworthy":    nw_text or None,
    }


# ── main loop ─────────────────────────────────────────────────────────────────

NEW_COLS = [
    "mobot_url", "mobot_match_title", "mobot_match_score",
    "mobot_common_name", "mobot_type", "mobot_family",
    "mobot_zone", "mobot_height", "mobot_spread",
    "mobot_bloom_time", "mobot_bloom_desc",
    "mobot_sun", "mobot_water", "mobot_maintenance",
    "mobot_suggested_use", "mobot_flower", "mobot_tolerate",
    "mobot_fall_color", "mobot_leaf_type", "mobot_sex_info",
    "mobot_noteworthy",
]

def _save_checkpoint(df, records):
    tmp = pd.DataFrame(records).set_index("index")
    for col in [c for c in NEW_COLS if c not in tmp.columns]:
        tmp[col] = None
    df.join(tmp[NEW_COLS], how="left").to_csv(CHECKPOINT_PATH, index=False)

def _merge_results(df, records):
    tmp = pd.DataFrame(records).set_index("index")
    for col in [c for c in NEW_COLS if c not in tmp.columns]:
        tmp[col] = None
    return df.join(tmp[NEW_COLS], how="left")

def run_scrape(df, delay=1.5, checkpoint_every=10, headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1280,900")
    driver = webdriver.Chrome(options=opts)

    records = []
    try:
        for i, (idx, row) in enumerate(df.iterrows()):
            tree_name = row["tree_name"]
            print(f"[{idx:3d}] {tree_name}", end=" ... ")

            pairs = get_search_results(driver, tree_name)
            title, url, score = pick_best(tree_name, pairs)
            print(f"score={score:.2f}  {title or 'NO MATCH'}")

            rec = {
                "index": idx,
                "mobot_url": url,
                "mobot_match_title": title,
                "mobot_match_score": score,
            }
            if url:
                detail = scrape_detail(url)
                if detail:
                    rec.update(detail)

            records.append(rec)
            time.sleep(delay)

            if (i + 1) % checkpoint_every == 0:
                _save_checkpoint(df, records)
                print(f"  >> checkpoint saved ({i + 1} rows done)")
    finally:
        driver.quit()

    return _merge_results(df, records)


In [ ]:
phenos_v2 = run_scrape(phenos, delay=1.5, headless=True)
phenos_v2.to_csv(OUTPUT_PATH, index=False)
phenos_v2